In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import loguniform, uniform
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report
import mlflow
import mlflow.sklearn

# Параметры для генерации даннных
STUDENT_ID = 796695 
CURRENT_YEAR = 2026
RANDOM_SEED = STUDENT_ID + CURRENT_YEAR

print(f"Seed для генерации: {RANDOM_SEED}")

### 2. Генерация синтетического датасета
Мы генерируем датасет со следующими параметрами:
*   Количество объектов: 400.
*   Количество признаков: 5 (из них 3 информативных и 2 шумовых).
*   Количество классов: 3.

Дополнительно, согласно заданию, мы вручную изменяем средние значения (`shifts`) и масштабы (`scales`) признаков так, чтобы они отличались друг от друга более чем на 50%. Полученные данные сохраняются в CSV-файл.

In [ ]:
# Генерируем 400 объектов, 5 признаков (3 информативных, 2 лишних)
X_raw, y_raw = make_classification(
    n_samples=400, 
    n_features=5, 
    n_informative=3, 
    n_redundant=2, 
    n_classes=3, 
    random_state=RANDOM_SEED
)

# Создаем разный масштаб и разные средние (отличия > 50%)
# Признак 1: масштаб 10, среднее -20
# Признак 2: масштаб 1, среднее 50 и так далее
shifts = np.array([-20, 50, 0, 10, -5])
scales = np.array([10.0, 1.0, 5.0, 0.5, 2.5])

X_final = (X_raw + shifts) * scales

# Формируем Датафрейм
columns = [f'feature_{i+1}' for i in range(5)]
df_init = pd.DataFrame(X_final, columns=columns)
df_init['target'] = y_raw

# Сохраняем в файл CSV
df_init.to_csv('dataset.csv', index=False)
print("Файл 'dataset.csv' создан.")

### 3. Загрузка данных и статистическое описание
Загружаем ранее созданный файл `dataset.csv`. Для первичного анализа распределения признаков используем **7-point summary** (функция `describe`), которая включает в себя: среднее, стандартное отклонение, минимум, максимум и квартили (25%, 50%, 75%).

In [ ]:
data = pd.read_csv('dataset.csv')

print("7-point summary (описание распределений):")
data.describe()

### 4. Корреляционный анализ
Оценим взаимосвязь признаков с помощью коэффициента корреляции Пирсона. Это позволит нам визуально определить, какие признаки имеют наибольшее влияние на целевую переменную (`target`), а какие являются малозначимыми (шумовыми).

In [ ]:
# Оценка влияния признаков на целевую переменную
corr_matrix = data.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', square=True)
plt.title("Матрица корреляции Пирсона")
plt.show()

print("Корреляция с целевой переменной:")
print(corr_matrix['target'].sort_values(ascending=False))

### 5. Анализ баланса классов
Для качественного обучения модели классификации важно знать, сбалансированы ли классы. Существенный перекос в сторону одного из классов может привести к смещению предсказаний модели.

In [ ]:
# Проверка сбалансированности выборки
plt.figure(figsize=(6, 4))
sns.countplot(data=data, x='target')
plt.title("Распределение классов в целевой переменной")
plt.xlabel("Класс")
plt.ylabel("Количество объектов")
plt.show()

# Вывод долей классов
print(data['target'].value_counts(normalize=True))

### 6. Подготовка данных к обучению
Разделяем данные на матрицу объект-признак (`X`) и вектор целевых значений (`y`). Осуществляем разбиение выборки на обучающую и тестовую в соотношении 80/20 с применением стратификации по целевой переменной.

In [ ]:
# Разделение на признаки и таргет
X = data.drop('target', axis=1)
y = data['target']

# Разбиение на обучающую и тестовую (20%) выборки со стратификацией
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

print(f"Размер обучающей выборки: {X_train.shape}")
print(f"Размер тестовой выборки: {X_test.shape}")

### 7. Построение модели: Логистическая регрессия
Первая модель — логистическая регрессия с регуляризацией. Мы используем `RandomizedSearchCV` для поиска оптимальных параметров: силы регуляризации (`C`) и типа штрафа (`penalty`). 

*Примечание:* Использование алгоритма `saga` обусловлено его эффективностью на больших выборках и поддержкой всех типов регуляризации.

**Параметры:**
* **`C`** — обратная сила регуляризации. Чем *меньше* значение, тем *сильнее* штраф за сложность модели (защита от переобучения).
* **`L1` (Lasso)** — может обнулять веса неважных признаков, работая как встроенный фильтр (отбор признаков).
* **`L2` (Ridge)** — плавно уменьшает веса всех признаков, не давая ни одному из них перетянуть влияние на себя. Хорошо работает при корреляции признаков.
* **`ElasticNet`** — комбинация L1 и L2: одновременно отбирает признаки и сохраняет стабильность модели.

In [ ]:
# Инициализация модели
lr_model = LogisticRegression(solver='saga', max_iter=5000)

# Пространство гиперпараметров
lr_params = {
    'C': loguniform(1e-3, 1e3),
    'penalty': ['l1', 'l2', 'elasticnet', None],
    'l1_ratio': uniform(0, 1),
    'class_weight': [None, 'balanced']
}

# Случайный поиск по сетке
lr_search = RandomizedSearchCV(
    lr_model, 
    param_distributions=lr_params, 
    n_iter=100, 
    cv=5, 
    scoring='accuracy', 
    random_state=RANDOM_SEED, 
    n_jobs=-1
)

lr_search.fit(X_train, y_train)

print(f"Лучшие параметры LogReg: {lr_search.best_params_}")
print(f"Accuracy на кросс-валидации: {lr_search.best_score_:.4f}")

In [ ]:
# MLflow отчет для RandomizedSearchCV (LogReg)
mlflow.set_experiment("ML_2026_Search_Reports")

with mlflow.start_run(run_name="LogReg_RandomSearch"):
    mlflow.log_params({
        "model_type": "LogisticRegression",
        "search_type": "RandomizedSearchCV",
        "train_size": len(y_train),
        "test_size": len(y_test),
        "features_n": X_train.shape[1]
    })

    mlflow.log_metric("best_cv_score", lr_search.best_score_)

    for param_name, param_value in lr_search.best_params_.items():
        mlflow.log_param(f"best_{param_name}", param_value)

    mlflow.sklearn.log_model(lr_search.best_estimator_, artifact_path="model")

### 8. Оценка качества логистической регрессии
Проверим качество модели на тестовых данных, которые она не видела при обучении. Основными метриками будут являться Precision, Recall и F1-score для каждого из трех классов.

In [ ]:
# Валидация на тестовых данных
lr_predictions = lr_search.predict(X_test)

print("Отчет по классификации (Logistic Regression) на тесте:")
print(classification_report(y_test, lr_predictions))

### 8-1. Плюсы и минусы логистической регрессии

### Плюсы

1. Понятная интерпретация влияния признаков на результат.
2. Выдает вероятность события, а не класс.
3. Быстро обучается и работает на данных.
4. Простая реализация и низкие системные требования.

### Минусы

1. Не находит сложные нелинейные зависимости данных.
2. Чувствительна к сильной корреляции между признаками.
3. Аномальные выбросы сильно снижают точность прогноза.
4. Требует тщательной предварительной подготовки всех признаков.

### 9. Построение модели: Метод k-ближайших соседей (k-NN)
Вторая модель — k-NN. Этот метод основан на измерении расстояния между объектами. С помощью `GridSearchCV` мы найдем наилучшее количество соседей и оптимальную метрику расстояния (Евклидова, Манхэттенская и др.).

In [ ]:
# Инициализация модели k-NN
knn_model = KNeighborsClassifier()

# Поиск по сетке (GridSearch)
knn_params = {
    'n_neighbors': range(1, 31, 2),
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan', 'minkowski']
}

knn_grid = GridSearchCV(
    knn_model, 
    param_grid=knn_params, 
    cv=5, 
    scoring='accuracy', 
    n_jobs=-1
)

knn_grid.fit(X_train, y_train)

print(f"Лучшие параметры k-NN: {knn_grid.best_params_}")
print(f"Accuracy на кросс-валидации: {knn_grid.best_score_:.4f}")

In [ ]:
# MLflow отчет для GridSearchCV (k-NN)
mlflow.set_experiment("ML_2026_Search_Reports")

with mlflow.start_run(run_name="KNN_GridSearch"):
    mlflow.log_params({
        "model_type": "KNeighborsClassifier",
        "search_type": "GridSearchCV",
        "train_size": len(y_train),
        "test_size": len(y_test),
        "features_n": X_train.shape[1]
    })

    mlflow.log_metric("best_cv_score", knn_grid.best_score_)

    for param_name, param_value in knn_grid.best_params_.items():
        mlflow.log_param(f"best_{param_name}", param_value)

    mlflow.sklearn.log_model(knn_grid.best_estimator_, artifact_path="model")

### 10. Оценка качества k-NN и общие выводы
Проводим финальную оценку k-NN. 

**Важное замечание по масштабированию:** 
В ходе анализа (этап 3) было выявлено, что признаки имеют существенно разный масштаб. Поскольку k-NN крайне чувствителен к расстояниям, в реальных задачах здесь рекомендуется применять `StandardScaler`. Однако, в рамках текущего исследования, параметры подбирались на исходных данных для оценки устойчивости алгоритмов к разным масштабам «из коробки».

In [ ]:
# Валидация на тестовых данных
best_knn = knn_grid.best_estimator_
knn_predictions = best_knn.predict(X_test)

print("Отчет по классификации (k-NN) на тесте:")
print(classification_report(y_test, knn_predictions))

### 11. Метод SVM

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report

svm = SVC(kernel='rbf', C = 0.001)

# C - связь с зазором между классами
# kernel - тип ядра ('linear', 'rbf', 'poly', 'sigmoid') определяет как преобразовывать данные в пространство более высокой размерности
# gamma - влияет на кривизну разделающей линии; малые значения - гладкие, большие - сложные

svm.fit(X_train, y_train)


print("Train")
y_train_pred = svm.predict( X_train )
print( classification_report( y_train, y_train_pred, digits=4 ) )

print()
print("Test")
y_test_pred = svm.predict( X_test )
print( classification_report( y_test, y_test_pred, digits=4 ) )

In [ ]:
from sklearn.model_selection import cross_val_score

ac = cross_val_score(svm, X, y, scoring ='accuracy')
ac.mean()

In [ ]:
from sklearn.model_selection import GridSearchCV

svm_gs = GridSearchCV( 
    estimator = svm,                                                  # модель
    param_grid = {  "kernel": ["linear", "poly", "rbf", "sigmoid"],   # 4
                    'C':       [ 10**i for i in range(-5,6)]           # 11
                },
    scoring = 'accuracy',                                 # метрика
    cv = None,                                            # кол-во частей для крос-валидации
    n_jobs = -1,                                          # подбирать паралельно с использованием оптимального числа
    verbose = 2
    )


svm_gs.fit( X_train, y_train)

In [ ]:
# MLflow отчет для GridSearchCV (SVM)
mlflow.set_experiment("ML_2026_Search_Reports")

with mlflow.start_run(run_name="SVM_GridSearch"):
    mlflow.log_params({
        "model_type": "SVC",
        "search_type": "GridSearchCV",
        "train_size": len(y_train),
        "test_size": len(y_test),
        "features_n": X_train.shape[1]
    })

    mlflow.log_metric("best_cv_score", svm_gs.best_score_)

    for param_name, param_value in svm_gs.best_params_.items():
        mlflow.log_param(f"best_{param_name}", param_value)

    mlflow.sklearn.log_model(svm_gs.best_estimator_, artifact_path="model")

In [ ]:
print("Лучшие параметры SVM:")
print(svm_gs.best_params_)

print("\nКачество на train:")
svm_train_pred = svm_gs.predict(X_train)
print(classification_report(y_train, svm_train_pred, digits=4))

print("Качество на test:")
svm_test_pred = svm_gs.predict(X_test)
print(classification_report(y_test, svm_test_pred, digits=4))

In [ ]:
# Дополнительно: можно посмотреть все значения quality по фолдам
# svm_gs.cv_results_['mean_test_score']

In [ ]:
# Отчет перенесен в предыдущую ячейку для цельности раздела SVM

In [ ]:
# Отчет перенесен в предыдущую ячейку для цельности раздела SVM

### 12. Обучение модели: дерево решений (Decision Tree)
Добавим модель дерева решений и подберем гиперпараметры с помощью `GridSearchCV`. 

**Что подбираем:**
* **`criterion`** - функция качества разбиения (`gini` или `entropy`).
* **`max_depth`** - максимальная глубина дерева (контроль переобучения).
* **`min_samples_split`** - минимальное число объектов для разбиения узла.
* **`min_samples_leaf`** - минимальное число объектов в листе.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Инициализация модели дерева решений
tree_model = DecisionTreeClassifier(random_state=RANDOM_SEED)

# Поиск по сетке (GridSearch)
# gini, entropy - гиперпараметры, определяющие критерий разбиения узлов дерева.
tree_params = {
    "criterion": ["gini", "entropy"],
    "max_depth": [3, 5, 7, 10, None],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 4, 8]
}

tree_grid = GridSearchCV(
    tree_model,
    param_grid=tree_params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

tree_grid.fit(X_train, y_train)

print(f"Лучшие параметры Decision Tree: {tree_grid.best_params_}")
print(f"Accuracy на кросс-валидации: {tree_grid.best_score_:.4f}")

In [ ]:
# MLflow отчет для GridSearchCV (Decision Tree)
mlflow.set_experiment("ML_2026_Search_Reports")

with mlflow.start_run(run_name="DecisionTree_GridSearch"):
    mlflow.log_params({
        "model_type": "DecisionTreeClassifier",
        "search_type": "GridSearchCV",
        "train_size": len(y_train),
        "test_size": len(y_test),
        "features_n": X_train.shape[1]
    })

    mlflow.log_metric("best_cv_score", tree_grid.best_score_)

    for param_name, param_value in tree_grid.best_params_.items():
        mlflow.log_param(f"best_{param_name}", param_value)

    mlflow.sklearn.log_model(tree_grid.best_estimator_, artifact_path="model")

### 13. Оценка качества дерева решений
Проверим качество лучшей версии дерева решений на тестовой выборке.

In [ ]:
# Проверка на тестовой выборке
best_tree = tree_grid.best_estimator_
tree_predictions = best_tree.predict(X_test)

print("Отчет по классификации (Decision Tree) на тесте:")
print(classification_report(y_test, tree_predictions))

### 14. Наивный байесовский классификатор
Обучим `GaussianNB`, затем подберем гиперпараметр `var_smoothing` с помощью `GridSearchCV`.

In [ ]:
from sklearn.naive_bayes import GaussianNB

# Gaussian - наивные байес строится, исходя из предположения, что признаки распределены по нормальному закону (Гаусса) и независимы друг от друга.
# Проверить гипотезу о нормальности распределения признаков.

nb_model = GaussianNB()
nb_model.fit(X_train, y_train)

nb_test_pred = nb_model.predict(X_test)

print("Отчет по классификации (GaussianNB) на тесте:")
print(classification_report(y_test, nb_test_pred, digits=4))

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score

nb_grid_params = {
    'var_smoothing': np.logspace(0, -9, num=100)
}

nb_search = GridSearchCV(
    estimator=GaussianNB(),
    param_grid=nb_grid_params,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

nb_search.fit(X_train, y_train)

nb_train_pred = nb_search.predict(X_train)
nb_test_pred = nb_search.predict(X_test)

nb_acc_train = accuracy_score(y_train, nb_train_pred)
nb_acc_test = accuracy_score(y_test, nb_test_pred)

print(f"Лучшие параметры GaussianNB: {nb_search.best_params_}")
print(f"Best CV accuracy: {nb_search.best_score_:.4f}")
print(f"Train accuracy: {nb_acc_train:.4f}")
print(f"Test accuracy: {nb_acc_test:.4f}")

# MLflow отчет для GridSearchCV (GaussianNB)
mlflow.set_experiment("ML_2026_Search_Reports")

with mlflow.start_run(run_name="GaussianNB_GridSearch"):
    mlflow.log_params({
        "model_type": "GaussianNB",
        "search_type": "GridSearchCV",
        "train_size": len(y_train),
        "test_size": len(y_test),
        "features_n": X_train.shape[1]
    })

    mlflow.log_param("grid_search_params", str(nb_grid_params))
    mlflow.log_metrics({
        "acc_train": nb_acc_train,
        "acc_test": nb_acc_test,
        "best_cv_score": nb_search.best_score_
    })

    mlflow.sklearn.log_model(nb_search.best_estimator_, artifact_path="model")

### 15. Ансамбль моделей
Соберем `VotingClassifier` из SVM, GaussianNB и DecisionTree, затем выполним подбор гиперпараметров.

In [ ]:
from sklearn.ensemble import VotingClassifier
from sklearn.svm import SVC

voting_models = [
    ('svm_model', SVC(kernel='rbf', C=0.001, probability=True, random_state=RANDOM_SEED)),
    ('nb_model', GaussianNB()),
    ('tree_model', DecisionTreeClassifier(max_depth=4, random_state=RANDOM_SEED))
]

voting_model = VotingClassifier(estimators=voting_models, voting='soft')
voting_model.fit(X_train, y_train)

print("Train")
print(classification_report(y_train, voting_model.predict(X_train), digits=4))
print("\nTest")
print(classification_report(y_test, voting_model.predict(X_test), digits=4))

In [ ]:
voting_grid_params = {
    'svm_model__C': [0.001, 0.01, 0.1, 1.0],
    'tree_model__max_depth': [3, 5, 7, None],
    'nb_model__var_smoothing': np.logspace(-7, -9, num=3)
}

voting_search = GridSearchCV(
    estimator=VotingClassifier(
        estimators=[
            ('svm_model', SVC(kernel='rbf', C=0.001, probability=True, random_state=RANDOM_SEED)),
            ('nb_model', GaussianNB(var_smoothing=1e-9)),
            ('tree_model', DecisionTreeClassifier(max_depth=3, criterion='entropy', random_state=RANDOM_SEED))
        ],
        voting='soft'
    ),
    param_grid=voting_grid_params,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

voting_search.fit(X_train, y_train)

voting_train_pred = voting_search.predict(X_train)
voting_test_pred = voting_search.predict(X_test)

voting_acc_train = accuracy_score(y_train, voting_train_pred)
voting_acc_test = accuracy_score(y_test, voting_test_pred)

print(f"Лучшие параметры VotingClassifier: {voting_search.best_params_}")
print(f"Best CV accuracy: {voting_search.best_score_:.4f}")
print(f"Train accuracy: {voting_acc_train:.4f}")
print(f"Test accuracy: {voting_acc_test:.4f}")

# MLflow отчет для GridSearchCV (VotingClassifier)
mlflow.set_experiment("ML_2026_Search_Reports")

with mlflow.start_run(run_name="VotingClassifier_GridSearch"):
    mlflow.log_params({
        "model_type": "VotingClassifier",
        "search_type": "GridSearchCV",
        "train_size": len(y_train),
        "test_size": len(y_test),
        "features_n": X_train.shape[1]
    })

    mlflow.log_param("grid_search_params", str(voting_grid_params))
    mlflow.log_metrics({
        "acc_train": voting_acc_train,
        "acc_test": voting_acc_test,
        "best_cv_score": voting_search.best_score_
    })

    mlflow.sklearn.log_model(voting_search.best_estimator_, artifact_path="model")

### 16. Беггинг

#### Обучение

In [ ]:
from sklearn.ensemble import BaggingClassifier

base_tree = DecisionTreeClassifier(max_depth=3, criterion='entropy', random_state=RANDOM_SEED)

bagging_model = BaggingClassifier(
    estimator=base_tree,
    n_estimators=150, # кол-во деревьев 
    n_jobs=-1,
    random_state=RANDOM_SEED
)

bagging_model.fit(X_train, y_train)

print("Train")
print(classification_report(y_train, bagging_model.predict(X_train), digits=4))
print("\nTest")
print(classification_report(y_test, bagging_model.predict(X_test), digits=4))

#### Поиск по сетке гиперпараметров

In [ ]:
bagging_grid_params = {
    'n_estimators': [50, 100, 150],
    'max_samples': [0.7, 0.8, 1.0],
    'estimator__max_depth': [3, 5, 7],
    'estimator__criterion': ['gini', 'entropy']
}

bagging_search = GridSearchCV(
    estimator=BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=RANDOM_SEED),
        random_state=RANDOM_SEED
    ),
    param_grid=bagging_grid_params,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

bagging_search.fit(X_train, y_train)

bagging_train_pred = bagging_search.predict(X_train)
bagging_test_pred = bagging_search.predict(X_test)

bagging_acc_train = accuracy_score(y_train, bagging_train_pred)
bagging_acc_test = accuracy_score(y_test, bagging_test_pred)

print(f"Лучшие параметры Bagging: {bagging_search.best_params_}")
print(f"Best CV accuracy: {bagging_search.best_score_:.4f}")
print(f"Train accuracy: {bagging_acc_train:.4f}")
print(f"Test accuracy: {bagging_acc_test:.4f}")

In [ ]:
# MLflow отчет для GridSearchCV (Bagging)
mlflow.set_experiment("ML_2026_Search_Reports")

with mlflow.start_run(run_name="Bagging_GridSearch"):
    mlflow.log_params({
        "model_type": "BaggingClassifier",
        "search_type": "GridSearchCV",
        "train_size": len(y_train),
        "test_size": len(y_test),
        "features_n": X_train.shape[1]
    })

    mlflow.log_param("grid_search_params", str(bagging_grid_params))
    mlflow.log_metrics({
        "acc_train": bagging_acc_train,
        "acc_test": bagging_acc_test,
        "best_cv_score": bagging_search.best_score_
    })

    mlflow.sklearn.log_model(bagging_search.best_estimator_, artifact_path="model")

### 17. Случайный лес

#### Обучение

In [ ]:
from sklearn.ensemble import RandomForestClassifier

forest = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    n_jobs=-1,
    random_state=RANDOM_SEED
)

forest.fit(X_train, y_train)

print("Train")
print(classification_report(y_train, forest.predict(X_train), digits=4))
print("\nTest")
print(classification_report(y_test, forest.predict(X_test), digits=4))

#### Поиск по сетке гиперпараметров

In [ ]:
rf_grid_params = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20], # заменить на [1, 2, 3]
    'min_samples_split': [2, 5, 10],
    'max_features': ['sqrt', 'log2']
}

rf_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=RANDOM_SEED),
    param_grid=rf_grid_params,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

rf_search.fit(X_train, y_train)

rf_train_pred = rf_search.predict(X_train)
rf_test_pred = rf_search.predict(X_test)

rf_acc_train = accuracy_score(y_train, rf_train_pred)
rf_acc_test = accuracy_score(y_test, rf_test_pred)

print(f"Лучшие параметры RandomForest: {rf_search.best_params_}")
print(f"Best CV accuracy: {rf_search.best_score_:.4f}")
print(f"Train accuracy: {rf_acc_train:.4f}")
print(f"Test accuracy: {rf_acc_test:.4f}")

In [ ]:
# MLflow отчет для GridSearchCV (Random Forest)
# 
mlflow.set_experiment("ML_2026_Search_Reports")

with mlflow.start_run(run_name="RandomForest_GridSearch"):
    mlflow.log_params({
        "model_type": "RandomForestClassifier",
        "search_type": "GridSearchCV",
        "train_size": len(y_train),
        "test_size": len(y_test),
        "features_n": X_train.shape[1]
    })

    mlflow.log_param("grid_search_params", str(rf_grid_params))
    mlflow.log_metrics({
        "acc_train": rf_acc_train,
        "acc_test": rf_acc_test,
        "best_cv_score": rf_search.best_score_
    })

    mlflow.sklearn.log_model(rf_search.best_estimator_, artifact_path="model")

### 18. Бустинг (CatBoost)

#### Обучение

In [ ]:
import catboost as cb

cat = cb.CatBoostClassifier(
    iterations=500,
    learning_rate=0.1,
    max_depth=4,
    task_type='CPU',
    random_seed=RANDOM_SEED,
    verbose=False
)

cat.fit(X_train, y_train)

print("Train")
print(classification_report(y_train, cat.predict(X_train), digits=4))
print("\nTest")
print(classification_report(y_test, cat.predict(X_test), digits=4))

#### Поиск по сетке гиперпараметров

In [ ]:
from typing import cast
from sklearn.base import BaseEstimator

cat_grid_params = {
    'iterations': [200, 500],
    'learning_rate': [0.03, 0.1],
    'depth': [1, 2, 3, 4],
    'l2_leaf_reg': [1, 3, 5]
}

cat_estimator = cast(
    BaseEstimator,
    cb.CatBoostClassifier(
        task_type='CPU',
        random_seed=RANDOM_SEED,
        verbose=True
    )
)

cat_search = GridSearchCV(
    estimator=cat_estimator,
    param_grid=cat_grid_params,
    cv=3,
    scoring='accuracy',
    n_jobs=-1
)

cat_search.fit(X_train, y_train)

cat_train_pred = cat_search.predict(X_train)
cat_test_pred = cat_search.predict(X_test)

cat_acc_train = accuracy_score(y_train, cat_train_pred)
cat_acc_test = accuracy_score(y_test, cat_test_pred)

print(f"Лучшие параметры CatBoost: {cat_search.best_params_}")
print(f"Best CV accuracy: {cat_search.best_score_:.4f}")
print(f"Train accuracy: {cat_acc_train:.4f}")
print(f"Test accuracy: {cat_acc_test:.4f}")

In [ ]:
# MLflow отчет для GridSearchCV (CatBoost)
mlflow.set_experiment("ML_2026_Search_Reports")

with mlflow.start_run(run_name="CatBoost_GridSearch"):
    mlflow.log_params({
        "model_type": "CatBoostClassifier",
        "search_type": "GridSearchCV",
        "train_size": len(y_train),
        "test_size": len(y_test),
        "features_n": X_train.shape[1]
    })

    mlflow.log_param("grid_search_params", str(cat_grid_params))
    mlflow.log_metrics({
        "acc_train": cat_acc_train,
        "acc_test": cat_acc_test,
        "best_cv_score": cat_search.best_score_
    })

    mlflow.sklearn.log_model(cat_search.best_estimator_, artifact_path="model")